# Stream Cipher

## 1. Generación de Keysteam

- ### Generador de números pseudoaleatorios (PRNG) básico.


In [2]:
import random
from datetime import datetime

def generate_prng_seed(seed=None):
    """
    Genera una semilla para el generador de números pseudoaleatorios (PRNG).
    
    Parámetros:
        seed (int, opcional): Si se proporciona, se usa como semilla. 
                              Si es None, se usa la marca de tiempo actual.
    
    Retorna:
        int: La semilla generada.
    """
    if seed is None:
        seed = int(datetime.now().timestamp())  # Usa la marca de tiempo actual si no hay semilla
    return seed

- ### Generación del Keystream


In [3]:
def generate_keystream(length, seed=None):
    """
    Genera un keystream pseudoaleatorio de una longitud dada.

    Parámetros:
        length (int): Longitud del keystream.
        seed (int, opcional): Semilla para el PRNG.

    Retorna:
        list: Una lista de bytes pseudoaleatorios (valores entre 0 y 255).
    """
    seed = generate_prng_seed(seed)  # Obtiene la semilla
    random.seed(seed)  # Inicializa el PRNG con la semilla

    return [random.randint(0, 255) for _ in range(length)]  # Genera el keystream

- ### Uso del Keystream en un mensaje


In [8]:
# Mensaje de prueba
message = "Hola, Stream Cipher!"  # Mensaje en texto plano

# Generar el keystream con una semilla específica
keystream = generate_keystream(len(message), seed=12345)

# Mostrar resultados
print("Mensaje:", message)
print("Keystream generado:", keystream)

Mensaje: Hola, Stream Cipher!
Keystream generado: [213, 5, 152, 188, 99, 138, 223, 82, 191, 63, 221, 133, 89, 95, 181, 46, 211, 85, 75, 105]


## 2. Cifrado

In [13]:
# Usando lo de Utility.py

def xor_message_with_keystream(message, keystream):
    """
    Cifra un mensaje utilizando XOR con un keystream dado.

    Parámetros:
        message (str): Mensaje en texto plano.
        keystream (list): Lista de números enteros (bytes) generados para el cifrado.

    Retorna:
        list: Lista de valores enteros representando el mensaje cifrado.
    """
    if len(message) != len(keystream):
        raise ValueError("El keystream debe tener la misma longitud que el mensaje.")

    message_bytes = [ord(char) for char in message]  # Convertir mensaje en valores ASCII
    encrypted_bytes = [m ^ k for m, k in zip(message_bytes, keystream)]  # XOR entre mensaje y keystream

    return encrypted_bytes

In [14]:
encrypted_message = xor_message_with_keystream(message=message, keystream=keystream)


print("Mensaje Original:", message)
print("Mensaje Cifrado (XOR):", encrypted_message)

Mensaje Original: Hola, Stream Cipher!
Mensaje Cifrado (XOR): [157, 106, 244, 221, 79, 170, 140, 38, 205, 90, 188, 232, 121, 28, 220, 94, 187, 48, 57, 72]


## 3. Decifrado

In [19]:
def decrypt_xor_message(encrypted_message, keystream):
    """
    Descifra un mensaje cifrado utilizando XOR con el mismo keystream.

    Parámetros:
        encrypted_message (list): Lista de valores enteros cifrados.
        keystream (list): Lista de números enteros (bytes) utilizados en el cifrado.

    Retorna:
        str: Mensaje original en texto plano.
    """
    if len(encrypted_message) != len(keystream):
        raise ValueError("El keystream debe tener la misma longitud que el mensaje cifrado.")

    # Aplicar XOR entre el mensaje cifrado y el keystream
    decrypted_bytes = [e ^ k for e, k in zip(encrypted_message, keystream)]

    # Convertir los valores ASCII a caracteres
    return ''.join(chr(b) for b in decrypted_bytes)

In [20]:
print("Mensaje Descifrado (XOR):", decrypt_xor_message(encrypted_message=encrypted_message, keystream=keystream))

Mensaje Descifrado (XOR): Hola, Stream Cipher!


## 4. Preguntas a responder

* ¿Qué sucede cuando cambias la clave utilizada para generar el keystream?

    * el keystream resultante será diferente. Esto significa que, aunque envíe el mismo mensaje, el texto cifrado será distinto cada vez que use una clave diferente

* ¿Qué riesgos de seguridad existen si reutilizas el mismo keystream para cifrar dos mensajes diferentes?
    
    *  usando el mismo keystream para cifrar dos mensajes distintos, un atacante podría comparar los dos textos cifrados y, mediante análisis, descubrir información sobre los mensajes originales. Esto se debe a que las similitudes entre los mensajes pueden reflejarse en los textos cifrados cuando se usa el mismo keystream


* ¿Cómo afecta la longitud del keystream a la seguridad del cifrado?

    * si el keystream es más corto y se reutiliza para partes del mensaje, esto puede crear patrones que los atacantes podrían detectar y aprovechar para descifrar el mensaje

* ¿Qué consideraciones debes tener al generar un keystream en un entorno real?

    * no reutilizar keystreams, cada mensaje debe cifrarse con un keystream único para evitar riesgos de seguridad
	* utilizar generadores de números aleatorios, para que el keystream sea impredecible y resistente a ataques
    * longitud del keystream, que debe de ser al menos tan largo como el mensaje para que no se encuentren patrones repetitivos

## Referencias 

* https://www.geeksforgeeks.org/pseudo-random-number-generator-prng/
* https://keepcoding.io/blog/que-es-flujo-de-claves/?utm_source=chatgpt.com
* GPT (ver archivo de ChatGPTPrompts.txt)